# Connection Test

Minimal notebook to create and test a Neo4j JDBC connection via Unity Catalog.
Designed to run on a **SQL Warehouse** (no custom libraries required).

In [ ]:
# Load configuration from secrets
SCOPE_NAME = "neo4j-uc-creds"

NEO4J_HOST = dbutils.secrets.get(SCOPE_NAME, "host")
NEO4J_USERNAME = dbutils.secrets.get(SCOPE_NAME, "user")
NEO4J_PASSWORD = dbutils.secrets.get(SCOPE_NAME, "password")

try:
    NEO4J_DATABASE = dbutils.secrets.get(SCOPE_NAME, "database")
except:
    NEO4J_DATABASE = "neo4j"

UC_CONNECTION_NAME = "aircraft_connection_v2"
JDBC_JAR_PATH = "/Volumes/main/jdbc_drivers/jars/neo4j-unity-catalog-connector-1.0.0-SNAPSHOT.jar"

NEO4J_JDBC_URL_SQL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/{NEO4J_DATABASE}?enableSQLTranslation=true"
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print(f"Connection Name: {UC_CONNECTION_NAME}")
print(f"JDBC URL: {NEO4J_JDBC_URL_SQL}")
print(f"JAR Path: {JDBC_JAR_PATH}")

In [ ]:
# List all existing connections
print("Existing connections:")
spark.sql("SHOW CONNECTIONS").show(truncate=False)

In [ ]:
# Drop existing connection
spark.sql(f"DROP CONNECTION IF EXISTS {UC_CONNECTION_NAME}")
print(f"Dropped (if existed): {UC_CONNECTION_NAME}")

In [ ]:
# Create connection
create_sql = f"""
CREATE CONNECTION {UC_CONNECTION_NAME} TYPE JDBC
ENVIRONMENT (
  java_dependencies '{JAVA_DEPENDENCIES}'
)
OPTIONS (
  url '{NEO4J_JDBC_URL_SQL}',
  user '{NEO4J_USERNAME}',
  password '{NEO4J_PASSWORD}',
  driver 'org.neo4j.jdbc.Neo4jDriver',
  externalOptionsAllowList 'dbtable,query,partitionColumn,lowerBound,upperBound,numPartitions,fetchSize,customSchema'
)
"""

try:
    spark.sql(create_sql)
    print(f"[PASS] Connection '{UC_CONNECTION_NAME}' created")
except Exception as e:
    print(f"[FAIL] Create connection failed: {e}")

In [ ]:
# Verify connection
try:
    spark.sql(f"DESCRIBE CONNECTION {UC_CONNECTION_NAME}").show(truncate=False)
except Exception as e:
    print(f"[FAIL] Describe connection failed: {e}")

In [ ]:
# Test with remote_query
try:
    df = spark.sql(f"""
        SELECT * FROM remote_query(
            '{UC_CONNECTION_NAME}',
            query => 'SELECT 1 AS test'
        )
    """)
    df.show()
    print("[PASS] remote_query() working")
except Exception as e:
    print(f"[FAIL] remote_query() failed: {e}")